# EconoSense AI - SageMaker Setup

This notebook sets up the AWS SageMaker environment for developing our economic analysis and forecasting models. We'll configure:
1. SageMaker session and IAM roles
2. Instance types and environment variables
3. Required dependencies
4. S3 bucket for data storage
5. Experiment tracking

Let's get started!

## 1. Set Up SageMaker Session and Role

First, we'll initialize the SageMaker session and set up the IAM role with necessary permissions.

In [ ]:
import boto3
import sagemaker
from sagemaker import get_execution_role

# Initialize the SageMaker session
sagemaker_session = sagemaker.Session()

# Get the SageMaker execution role
role = get_execution_role()
print(f"SageMaker Role ARN: {role}")

# Initialize AWS clients
s3_client = boto3.client('s3')
sm_client = boto3.client('sagemaker')

## 2. Configure Instance Type and Environment

Now we'll set up the training instance type and configure environment variables for our ML workloads.

In [ ]:
# Define training instance configuration
training_instance_type = 'ml.m5.xlarge'  # Good balance of compute and cost
training_instance_count = 1

# Set environment variables
import os
os.environ['AWS_DEFAULT_REGION'] = sagemaker_session.boto_region_name
os.environ['SAGEMAKER_INSTANCE_TYPE'] = training_instance_type

print(f"AWS Region: {os.environ['AWS_DEFAULT_REGION']}")
print(f"Training Instance Type: {training_instance_type}")

# Define project name for resource tagging
project_name = 'econosense-ai'

## 3. Install Required Dependencies

Let's install and import all the necessary Python packages for our ML development.

In [ ]:
# Install required packages
!pip install --quiet \
    pandas \
    numpy \
    scikit-learn \
    torch \
    transformers \
    plotly \
    matplotlib \
    seaborn \
    pandas-datareader \
    yfinance \
    statsmodels

# Import common libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import torch
import plotly.express as px

print("All required packages have been installed and imported successfully!")

## 4. Create S3 Bucket for Data Storage

Set up an S3 bucket for storing our training data, models, and artifacts.

In [ ]:
# Set up S3 bucket for data storage
import boto3
from botocore.exceptions import ClientError

def create_bucket_if_not_exists(bucket_name, region=None):
    try:
        if region is None:
            s3_client = boto3.client('s3')
            s3_client.create_bucket(Bucket=bucket_name)
        else:
            s3_client = boto3.client('s3', region_name=region)
            location = {'LocationConstraint': region}
            s3_client.create_bucket(Bucket=bucket_name,
                                  CreateBucketConfiguration=location)
    except ClientError as e:
        if e.response['Error']['Code'] == 'BucketAlreadyOwnedByYou':
            print(f"Bucket {bucket_name} already exists.")
        else:
            raise e
    
    return f"s3://{bucket_name}"

# Create bucket for our project
bucket_name = f"{project_name}-{sagemaker_session.boto_region_name}"
bucket_uri = create_bucket_if_not_exists(bucket_name, region=sagemaker_session.boto_region_name)

# Create folder structure in the bucket
folder_structure = ['raw-data', 'processed-data', 'models', 'outputs']
for folder in folder_structure:
    s3_client.put_object(Bucket=bucket_name, Key=f"{folder}/")

print(f"S3 bucket created: {bucket_uri}")
print("Created folder structure:", folder_structure)

## 5. Set Up SageMaker Experiment Tracking

Configure SageMaker experiments to track our training jobs, metrics, and parameters.

In [ ]:
# Set up experiment tracking
from sagemaker.experiments.experiment import Experiment
from datetime import datetime

# Create an experiment
experiment_name = f"{project_name}-experiment-{datetime.now().strftime('%Y-%m-%d')}"
try:
    experiment = Experiment.create(
        experiment_name=experiment_name,
        description="Economic forecasting and sentiment analysis models",
        sagemaker_session=sagemaker_session
    )
except Exception as e:
    print(f"Experiment {experiment_name} already exists or error occurred: {str(e)}")
    experiment = Experiment.load(experiment_name=experiment_name)

print(f"Experiment created: {experiment_name}")

# Create a trial
trial_name = f"{project_name}-trial-{datetime.now().strftime('%Y-%m-%d-%H-%M')}"
trial = experiment.create_trial(trial_name=trial_name)

print(f"Trial created: {trial_name}")

# Save experiment and trial names for future use
experiment_config = {
    'experiment_name': experiment_name,
    'trial_name': trial_name,
    'bucket_name': bucket_name,
    'region': sagemaker_session.boto_region_name
}

print("\nSageMaker environment is ready for ML development!")
print(f"Configuration: {experiment_config}")